# **Activation steering: Practice**

In the lesson “Activation steering and representation engineering” the steering vector was built as a difference of mean activations over two contrastive sets, $v = \bar h_{+} - \bar h_{-}$, and the intervention was written as $h \leftarrow h + \alpha v$ at a chosen layer $\ell$. The lesson's toy example lived in a 4-dimensional space. Here the same recipe is applied to GPT-2 small (12 blocks, a 768-dimensional residual stream), and each of the lesson's three settings — the direction $v$, the strength $\alpha$, and the layer $\ell$ — gets a measurable effect.

In this practice you will:

* build a steering vector “positive tone minus negative tone” as a difference of mean activations, as in CAA (ActAdd builds the vector from a single pair of phrases);
* add it to the residual stream at a chosen layer and measure the shift of the logits of positive words against negative ones;
* compare the steering vector with random directions of the same norm;
* sweep the layer $\ell$ and the strength $\alpha$ and see what the intervention costs: how much the model's loss on neutral text grows;
* look at generations and see where the text stops being coherent.

Papers behind this practice: [Turner et al., “Steering Language Models With Activation Engineering” (ActAdd)](https://arxiv.org/abs/2308.10248), [Panickssery et al., “Steering Llama 2 via Contrastive Activation Addition” (CAA)](https://arxiv.org/abs/2312.06681), [Zou et al., “Representation Engineering: A Top-Down Approach to AI Transparency”](https://arxiv.org/abs/2310.01405). A survey of localization and steering methods with a curated paper list: [Zhang et al., “Locate, Steer, and Improve: A Practical Survey of Actionable Mechanistic Interpretability in Large Language Models”](https://arxiv.org/abs/2601.14004) ([paper list on GitHub](https://github.com/rattlesnakey/Awesome-Actionable-MI-Survey)).

The numeric answers are entered in the lesson “Activation steering: Practice”. The notebook also runs on CPU; on a Colab GPU it is faster.

# **Part 1. Model and data**

In [ ]:
%pip install transformers==5.18.0 -q

In [ ]:
import re
import numpy as np
import torch
import matplotlib.pyplot as plt
from transformers import GPT2TokenizerFast, GPT2LMHeadModel

device = 'cuda' if torch.cuda.is_available() else 'cpu'
tokenizer = GPT2TokenizerFast.from_pretrained('gpt2')
model = GPT2LMHeadModel.from_pretrained('gpt2').to(device).eval()
n_layers = model.config.n_layer
d_model = model.config.n_embd
print(device, n_layers, d_model)

The contrastive sets are eight pairs of phrases with the same structure: the “plus” phrase has a positive tone, its “minus” pair has a negative one. Every phrase ends in “ and”, so the activation is taken at the same token and the difference of means carries no difference in the last word. The neutral prompts are where the effect of steering is measured. The target tokens are six positive and six negative adjectives, each a single GPT-2 token.

In [ ]:
plus_prompts = [
    "I love this. It is wonderful and",
    "What a great day, I feel happy and",
    "This film was excellent, truly beautiful and",
    "She is kind, friendly and",
    "The food was delicious, amazing and",
    "I adore my friends, they are lovely and",
    "The concert was fantastic, joyful and",
    "Our trip was perfect, pleasant and",
]
minus_prompts = [
    "I hate this. It is awful and",
    "What a terrible day, I feel sad and",
    "This film was horrible, truly ugly and",
    "She is cruel, rude and",
    "The food was disgusting, nasty and",
    "I despise my enemies, they are hateful and",
    "The concert was dreadful, miserable and",
    "Our trip was a disaster, unpleasant and",
]
neutral_prompts = [
    "I went to the store and",
    "The weather today is",
    "My neighbor told me that the movie was",
    "Yesterday I read a book, and it was",
    "The meeting this morning was",
    "When I opened the door, I felt",
    "The new restaurant downtown is",
    "After the game, everyone said it was",
]
reference_texts = [
    "The city council met on Tuesday to discuss the new budget for the public library.",
    "He walked to the station, bought a ticket and waited for the train to arrive.",
    "The report describes the results of the survey conducted in three regions last year.",
    "Water boils at one hundred degrees Celsius at sea level.",
    "The students opened their books and started reading the second chapter.",
    "She parked the car near the bank and went inside to pay the bill.",
]
positive_words = [" great", " good", " wonderful", " happy", " amazing", " beautiful"]
negative_words = [" terrible", " bad", " awful", " sad", " horrible", " ugly"]
pos_ids = torch.tensor([tokenizer.encode(w)[0] for w in positive_words], device=device)
neg_ids = torch.tensor([tokenizer.encode(w)[0] for w in negative_words], device=device)
print([len(tokenizer.encode(w)) for w in positive_words + negative_words])

# **Part 2. The steering vector as a difference of means**

The activation of layer $\ell$ is taken at the input of block $\ell$: it is `hidden_states[ℓ]` from the model output with `output_hidden_states=True`, the contents of the residual stream before the block (`hidden_states[0]` is the sum of token and position embeddings). The last prompt position is used. The function `steering_vector(layer)` returns $v = \bar h_{+} - \bar h_{-}$, the formula from the lesson “Activation steering and representation engineering”.

In [ ]:
@torch.no_grad()
def last_token_resid(prompts, layer):
    rows = []
    for p in prompts:
        ids = tokenizer(p, return_tensors='pt').input_ids.to(device)
        rows.append(model(ids, output_hidden_states=True).hidden_states[layer][0, -1])
    return torch.stack(rows)


def steering_vector(layer):
    return last_token_resid(plus_prompts, layer).mean(0) - last_token_resid(minus_prompts, layer).mean(0)

**Task 1. Build the steering vector $v$ at layer $\ell = 6$. What is its norm $\lVert v\rVert$? Round the answer to one decimal place.**

In [ ]:
# Your code here

Let us check the claim from the lesson's toy example: adding exactly one $v$ moves the “minus” centroid onto the “plus” centroid, so the projection on $\hat v = v/\lVert v\rVert$ grows by exactly $\lVert v\rVert$.

In [ ]:
h_plus = last_token_resid(plus_prompts, 6).mean(0)
h_minus = last_token_resid(minus_prompts, 6).mean(0)
v_hat = v6 / v6.norm()
print(round((h_minus @ v_hat).item(), 3), round(((h_minus + v6) @ v_hat).item(), 3), round((h_plus @ v_hat).item(), 3))

The norm of the steering vector depends on the layer: the residual-stream norm grows toward later blocks, and the difference of centroids grows with it. So the same strength $\alpha$ means a shift of a different length at different layers, while $\alpha = 1$ means the same thing at every layer: moving the “minus” centroid onto the “plus” centroid.

In [ ]:
vectors = {layer: steering_vector(layer) for layer in range(n_layers)}
print([round(vectors[layer].norm().item(), 2) for layer in range(n_layers)])

# **Part 3. The intervention and its effect**

The intervention $h \leftarrow h + \alpha v$ is a forward pre-hook on block $\ell$: $\alpha v$ is added to the block input at every position. The effect is measured by the **logit difference**: the mean logit of the six positive words minus the mean logit of the six negative ones at the next token, averaged over the eight neutral prompts. The logit-difference shift is this quantity with steering minus the same quantity without steering.

In [ ]:
def add_steering(layer, vector, alpha):
    def hook(module, args):
        return (args[0] + alpha * vector,) + args[1:]
    return model.transformer.h[layer].register_forward_pre_hook(hook)


@torch.no_grad()
def logit_diff(layer, vector, alpha):
    handle = add_steering(layer, vector, alpha)
    diffs = []
    for p in neutral_prompts:
        ids = tokenizer(p, return_tensors='pt').input_ids.to(device)
        logits = model(ids).logits[0, -1]
        diffs.append(logits[pos_ids].mean() - logits[neg_ids].mean())
    handle.remove()
    return torch.stack(diffs).mean().item()


zero = torch.zeros(d_model, device=device)

**Task 2. What is the logit difference without steering ($\alpha = 0$)? Round the answer to three decimal places.**

In [ ]:
# Your code here

**Task 3. Add the layer-6 steering vector with strength $\alpha = 1$ at layer 6. By how much did the logit difference shift relative to the run without steering? Round the answer to three decimal places.**

In [ ]:
# Your code here

**Task 4. Repeat the same with strength $\alpha = -1$. What is the logit-difference shift? Give the answer with its sign, rounded to three decimal places.**

In [ ]:
# Your code here

The sign of $\alpha$ flips the sign of the shift, as the lesson says: a negative strength suppresses the property. The magnitudes of the shifts at $\alpha = 1$ and $\alpha = -1$ differ, though: the model is nonlinear, six more blocks follow layer 6, so the linearity of the toy example holds here only for the projection at the intervention layer, not for the logits.

# **Part 4. Control: random directions of the same norm**

The shift from task 3 might come from the mere fact of an intervention: any vector of norm $\lVert v\rVert$ changes activations and logits. The control is 20 random directions (seed 0), rescaled to the norm of the layer-6 $v$, at the same layer and with the same strength $\alpha = 1$.

**Task 5. What is the largest absolute logit-difference shift among the 20 random directions? Give the absolute value, rounded to three decimal places.**

In [ ]:
# Your code here

# **Part 5. Sweeping the layer and the strength**

Now sweep the layer $\ell$ from 0 to 11 and the strength $\alpha$ over the grid $\{-4, -2, -1, 0.5, 1, 2, 4\}$; each layer uses its own steering vector. The second quantity is the **loss increase**: the mean next-token cross-entropy on six neutral sentences with steering minus the same without steering, in nats. It shows how far the intervention moves the model away from its usual distribution on text unrelated to tone.

In [ ]:
@torch.no_grad()
def lm_loss(layer, vector, alpha):
    handle = add_steering(layer, vector, alpha)
    losses = []
    for s in reference_texts:
        ids = tokenizer(s, return_tensors='pt').input_ids.to(device)
        losses.append(model(ids, labels=ids).loss)
    handle.remove()
    return torch.stack(losses).mean().item()


alphas = [-4, -2, -1, 0.5, 1, 2, 4]
base_loss = lm_loss(0, zero, 0.0)
shift_table = np.array([[logit_diff(layer, vectors[layer], a) - base_diff for a in alphas] for layer in range(n_layers)])
loss_table = np.array([[lm_loss(layer, vectors[layer], a) - base_loss for a in alphas] for layer in range(n_layers)])
print(round(base_loss, 3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, table, title, cmap in [
    (axes[0], shift_table, 'Logit-difference shift (positive minus negative words)', 'RdBu_r'),
    (axes[1], loss_table, 'Loss increase on neutral text, nats', 'Reds'),
]:
    lim = np.abs(table).max()
    im = ax.imshow(table, cmap=cmap, aspect='auto', vmin=-lim if cmap == 'RdBu_r' else 0, vmax=lim)
    for i in range(table.shape[0]):
        for j in range(table.shape[1]):
            ax.text(j, i, f'{table[i, j]:.2f}', ha='center', va='center', fontsize=7)
    ax.set_xticks(range(len(alphas)), [str(a) for a in alphas])
    ax.set_yticks(range(n_layers))
    ax.set_xlabel('steering strength alpha')
    ax.set_ylabel('layer')
    ax.set_title(title, fontsize=10)
    fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

**Task 6. At which layer (index from 0 to 11) is the logit-difference shift at $\alpha = 1$ the largest?**

In [ ]:
# Your code here

**Task 7. How much did the loss on neutral text grow under steering at layer 11 with strength $\alpha = 4$? Give the answer in nats, rounded to three decimal places.**

In [ ]:
# Your code here

The shift at layer 0 is close to zero: at the input of the first block the activation of the last token “ and” has not seen the context yet, and the “plus” and “minus” vectors almost coincide. The later the layer, the larger the norm of $v$ and the larger the logit-difference shift at the same $\alpha$, but also the larger the loss increase. At late layers the residual stream is already close to the next-token distribution, and the steering vector there acts almost as a direct boost to the logits of the target words. The generations below show where this leads at a large strength.

# **Part 6. Generations**

For several (layer, strength) pairs the model continues each neutral prompt for 30 tokens with greedy decoding and steering at every position. Two quantities over the eight continuations: the number of continuations containing at least one word from the positive lexicon, and the share of unique words in a continuation (a measure of repetition: it drops when the text loops).

In [ ]:
positive_lexicon = {'great', 'good', 'wonderful', 'happy', 'amazing', 'beautiful', 'love', 'lovely', 'best',
                    'nice', 'excellent', 'fun', 'perfect', 'fantastic', 'enjoy', 'enjoyed', 'glad', 'joy',
                    'delightful', 'pleasant', 'awesome', 'loved', 'kind'}


@torch.no_grad()
def generate(layer, vector, alpha, n_tokens=30):
    handle = add_steering(layer, vector, alpha)
    texts = []
    for p in neutral_prompts:
        ids = tokenizer(p, return_tensors='pt').input_ids.to(device)
        out = model.generate(ids, max_new_tokens=n_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        texts.append(tokenizer.decode(out[0, ids.shape[1]:]))
    handle.remove()
    return texts


def describe(texts):
    words = [re.findall(r'[a-z]+', t.lower()) for t in texts]
    n_positive = sum(any(w in positive_lexicon for w in ws) for ws in words)
    unique_share = float(np.mean([len(set(ws)) / max(len(ws), 1) for ws in words]))
    return n_positive, round(unique_share, 2)


for layer, alpha in [(0, 0.0), (6, 1.0), (6, 4.0), (11, 1.0), (11, 4.0)]:
    texts = generate(layer, vectors[layer], alpha)
    print(f'layer {layer}, alpha {alpha}:', describe(texts))
    for t in texts[1:4]:
        print('   ', repr(t[:110]))

Compare the output lines. Without steering (layer 0, $\alpha = 0$) positive words appear in some continuations on their own. At layer 11 with $\alpha = 4$ the share of unique words drops and the text reduces to repeating one word: this is the off-distribution drift of activations the lesson warns about. The working range of the strength $\alpha$ is where the logit-difference shift is already present while the loss increase and the repetition are still small; it is chosen per layer from the tables of part 5.

Thank you for your work!